In [4]:
import os
import zipfile
import gzip
import io
from fitparse import FitFile
import gpxpy
import pandas as pd
import numpy as np

def load_activities_from_folder(folder):
    activities = []
    
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        file_lower = file.lower()
        
        try:
            # ----- GPX -----
            if file_lower.endswith('.gpx'):
                with open(path, "r", encoding="utf-8") as gpx_file:
                    gpx = gpxpy.parse(gpx_file)
                
                for track in gpx.tracks:
                    for segment in track.segments:
                        if not segment.points:
                            continue
                        start_time = segment.points[0].time
                        end_time = segment.points[-1].time
                        duration_min = (end_time - start_time).total_seconds() / 60
                        distance_km = segment.length_2d() / 1000
                        
                        activities.append({
                            "date": start_time,
                            "duration_min": duration_min,
                            "distance_km": distance_km,
                            "avg_hr": np.nan,          # No HR in GPX
                            "source": "gpx",
                            "file": file
                        })
            
            # ----- Plain .fit -----
            elif file_lower.endswith('.fit'):
                fitfile = FitFile(path)
                data = extract_fit_data(fitfile, file)
                if data:
                    activities.append(data)
            
            # ----- Zipped .fit (.zip or .fit.zip) -----
            elif file_lower.endswith(('.zip', '.fit.zip')):
                with zipfile.ZipFile(path) as z:
                    for name in z.namelist():
                        if name.lower().endswith('.fit'):
                            with z.open(name) as f:
                                fit_data = f.read()
                            fitfile = FitFile(io.BytesIO(fit_data))
                            data = extract_fit_data(fitfile, file)
                            if data:
                                activities.append(data)
                            break  # Assume one .fit per zip
            
            # ----- Gzipped .fit (.gz or .fit.gz) -----
            elif file_lower.endswith(('.gz', '.fit.gz')):
                with gzip.open(path, 'rb') as f:
                    fit_data = f.read()
                fitfile = FitFile(io.BytesIO(fit_data))
                data = extract_fit_data(fitfile, file)
                if data:
                    activities.append(data)
        
        except Exception as e:
            print(f"Error processing {file}: {e}")
    
    return activities

# Helper to extract from FIT
def extract_fit_data(fitfile, filename):
    timestamps = []
    heart_rates = []
    distances = []
    
    for record in fitfile.get_messages("record"):
        time = record.get_value("timestamp")
        hr = record.get_value("heart_rate")
        dist = record.get_value("distance")
        
        if time: timestamps.append(time)
        if hr: heart_rates.append(hr)
        if dist: distances.append(dist)
    
    if not timestamps:
        return None
    
    start_time = min(timestamps)
    end_time = max(timestamps)
    duration_min = (end_time - start_time).total_seconds() / 60
    avg_hr = np.mean(heart_rates) if heart_rates else np.nan
    total_distance_km = distances[-1] / 1000 if distances else np.nan
    
    return {
        "date": start_time,
        "duration_min": duration_min,
        "distance_km": total_distance_km,
        "avg_hr": avg_hr,
        "source": "fit",
        "file": filename
    }

In [ ]:
folder = r"C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis\data\activities"  # adjust if needed

activities = load_activities_from_folder(folder)
df = pd.DataFrame(activities)

# Clean & prepare
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True).dt.tz_localize(None)
df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

df['weekday'] = df['date'].dt.day_name()
df['week'] = df['date'].dt.to_period('W')   # ISO week (Mon start); change to 'W-FRI' if Vitality weeks end Friday

print(f"Total activities: {len(df)}")
print(df['source'].value_counts())          # See GPX vs FIT split
df.head()

KeyError: 'date'

In [ ]:

# activities = load_activities_from_folder(folder)
# df = pd.DataFrame(activities)

# Clean dates (as before)
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True).dt.tz_localize(None)
df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

df['weekday'] = df['date'].dt.day_name()
df['week'] = df['date'].dt.to_period('W')   # or 'W-FRI' if Vitality weeks end Friday

# Create the two views
df_all = df.copy()                          # Everything → duration, distance, counts
df_fit = df[df['source'] == 'fit'].copy()   # Only FIT → has avg_hr

print(f"Total activities (all): {len(df_all)}")
print(f"FIT activities (with HR): {len(df_fit)}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Workout frequency by day of week
plt.figure(figsize=(10, 6))
sns.countplot(data=df_all, 
              x='weekday', 
              order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.title("Workout Frequency by Day of Week (All Activities – GPX + FIT)")
plt.xlabel("Day")
plt.ylabel("Number of Workouts")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Average workout duration by day
plt.figure(figsize=(10, 6))
sns.barplot(data=df_all, 
            x='weekday', 
            y='duration_min', 
            order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
            errorbar=None)  # or 'ci' if you want confidence intervals
plt.title("Average Workout Duration by Day of Week (All Activities)")
plt.xlabel("Day")
plt.ylabel("Average Duration (minutes)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Weekly total duration over years
weekly_all = df_all.groupby('week').agg({
    'duration_min': 'sum',
    'distance_km': 'sum',
    'date': 'count'
}).rename(columns={'date': 'num_workouts'})

weekly_all.index = weekly_all.index.to_timestamp()
weekly_all = weekly_all.sort_index()

plt.figure(figsize=(14, 7))
plt.plot(weekly_all.index, weekly_all['duration_min'], color='teal', linewidth=1.8)
plt.title("Total Weekly Exercise Duration (2014–2026) – All Activities")
plt.ylabel("Total Minutes per Week")
plt.xlabel("Week Starting")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Points estimation (only possible for FIT)
YOUR_AGE = 35               # ← update to your age
MAX_HR = 220 - YOUR_AGE

def estimate_points(duration_min, avg_hr):
    if pd.isna(avg_hr) or duration_min < 15:
        return 0
    intensity = avg_hr / MAX_HR
    # ... (your full estimate_points function from earlier)
    # return the points value

df_fit['estimated_points'] = df_fit.apply(
    lambda row: estimate_points(row['duration_min'], row['avg_hr']),
    axis=1
)

# Weekly points (only from FIT activities)
weekly_fit = df_fit.groupby('week').agg({
    'estimated_points': 'sum',
    'duration_min': 'sum',          # optional: compare duration in HR-tracked workouts
}).rename(columns={'duration_min': 'fit_duration_min'})

weekly_fit.index = weekly_fit.index.to_timestamp()
weekly_fit = weekly_fit.sort_index()

# Plot weekly estimated points
plt.figure(figsize=(14, 7))
plt.plot(weekly_fit.index, weekly_fit['estimated_points'], color='purple', linewidth=1.8)
plt.axhline(900, color='red', linestyle='--', label='Typical 900-point Goal')
plt.title("Weekly Estimated Vitality Points (FIT / HR-tracked Activities Only)")
plt.ylabel("Estimated Points")
plt.xlabel("Week Starting")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()